In [ ]:
#| default_exp handlers.pipeline.assembly

#| export
from __future__ import annotations
from pathlib import Path
from .callbacks import PipelineState, run_pipeline
from .contracts import HandlerConfig
from .gates import gap_check


In [ ]:
#| export
from __future__ import annotations
from pathlib import Path
from marisco.callbacks import AddSampleIDCB, EncodeTimeCB, LowerStripNameCB, SanitizeLonLatCB
from marisco.handlers.pipeline.callbacks import (
    PipelineState, RenameColsCB, SoftConvertUnitCB, SoftMeltWideNuclidesCB,
    SoftParseDateTimeCB, SoftRemapCB, _GuardedEncodeTimeCB,
    _GuardedSanitizeLonLatCB, run_pipeline,
)
from marisco.handlers.pipeline.contracts import HandlerConfig
from marisco.handlers.pipeline.gates import gap_check

## Assembly Helpers

In [ ]:
#| export
def build_normalize_cbs(cfg: HandlerConfig) -> list:
    "Assemble the normalization callbacks declared in normalize_case."
    return [LowerStripNameCB(col_src=s, col_dst=d) for s, d in cfg.normalize_case.items()]


def build_chain(cfg: HandlerConfig) -> list:
    "Assemble the legacy single-phase callback chain from a handler config."
    col_provider = next((v for v in cfg.rename.values() if v.endswith('_PROVIDER')), None)
    return [
        RenameColsCB(mapping=cfg.rename, string_cast=cfg.string_cast),
        SoftParseDateTimeCB(col_date=cfg.col_date, col_time=cfg.col_time, fmt=cfg.dt_format),
        SoftMeltWideNuclidesCB(spec=[s.model_dump() for s in cfg.melt_spec]),
        *[SoftConvertUnitCB(rule=r.model_dump()) for r in cfg.unit_conversions],
        SoftRemapCB(col_src='NUCLIDE', col_remap='NUCLIDE', lut=cfg.nuclide_lut),
        SoftRemapCB(col_src='UNIT',    col_remap='UNIT',    lut=cfg.unit_lut),
        SoftRemapCB(col_src='LAB',     col_remap='LAB',     lut=cfg.lab_lut),
        SoftRemapCB(col_src='NUCLIDE', col_remap='AREA',    lut={}, default_val=cfg.area_default),
        SanitizeLonLatCB(),
        EncodeTimeCB(),
        AddSampleIDCB(col_provider=col_provider),
    ]


def build_phase_pipelines(cfg: HandlerConfig) -> tuple[list, list]:
    "Split the standard core chain into pre-lossy and post-lossy phases for deep Gate 2."
    merged = cfg.model_copy(update={
        "rename": {**cfg.columns, **cfg.rename},
        "dt_format": cfg.time_format or cfg.dt_format,
    })
    col_provider = next((v for v in merged.rename.values() if v.endswith('_PROVIDER')), None)

    pre_lossy = [
        RenameColsCB(mapping=merged.rename, string_cast=merged.string_cast),
        SoftParseDateTimeCB(col_date=merged.col_date, col_time=merged.col_time, fmt=merged.dt_format),
        SoftMeltWideNuclidesCB(spec=[s.model_dump() for s in merged.melt_spec]),
        *[SoftConvertUnitCB(rule=r.model_dump()) for r in merged.unit_conversions],
        SoftRemapCB(col_src='NUCLIDE', col_remap='NUCLIDE', lut=merged.nuclide_lut),
        SoftRemapCB(col_src='UNIT',    col_remap='UNIT',    lut=merged.unit_lut),
        SoftRemapCB(col_src='LAB',     col_remap='LAB',     lut=merged.lab_lut),
        SoftRemapCB(col_src='NUCLIDE', col_remap='AREA',    lut={}, default_val=merged.area_default),
    ]
    post_lossy = [
        _GuardedSanitizeLonLatCB(),
        _GuardedEncodeTimeCB(),
        AddSampleIDCB(col_provider=col_provider),
    ]
    return pre_lossy, post_lossy


def build_core_pipeline(cfg: HandlerConfig) -> list:
    "Auto-assemble the standard core callback chain with topology guards."
    pre_lossy, post_lossy = build_phase_pipelines(cfg)
    return [*pre_lossy, *post_lossy]


def run_preflight(state: PipelineState, cfg: HandlerConfig, yaml_dir: Path = None) -> PipelineState:
    "Run all declarative, non-lossy transformations and Gate 2 in memory."
    pre_lossy, _ = build_phase_pipelines(cfg)
    chain = [
        *build_normalize_cbs(cfg),
        *pre_lossy,
    ]
    run_pipeline(state, chain)
    gap_check(cfg, state.dfs)
    return state


def run_finalize(state: PipelineState, cfg: HandlerConfig, yaml_dir: Path = None) -> PipelineState:
    "Run the lossy/output-facing phase after preflight has passed."
    _, post_lossy = build_phase_pipelines(cfg)
    run_pipeline(state, post_lossy)
    return state
